In [1]:
import pandas as pd
import numpy as np

This is for testing the baseline_model and to identify area for feature engineering

In [2]:
df1 = pd.read_csv("../data/materials_with_elasticity.csv")
y = df1["bulk_modulus_vrh"]
X = pd.read_csv("../data/materials_data.csv")

In [3]:
X.isnull().sum()

MagpieData minimum Number              0
MagpieData maximum Number              0
MagpieData range Number                0
MagpieData mean Number                 0
MagpieData avg_dev Number              0
                                      ..
MagpieData maximum SpaceGroupNumber    0
MagpieData range SpaceGroupNumber      0
MagpieData mean SpaceGroupNumber       0
MagpieData avg_dev SpaceGroupNumber    0
MagpieData mode SpaceGroupNumber       0
Length: 132, dtype: int64

In [5]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 132 entries, MagpieData minimum Number to MagpieData mode SpaceGroupNumber
dtypes: float64(132)
memory usage: 5.0 MB


In [6]:
print(y.describe())

count    5000.000000
mean      106.617061
std        69.558712
min         0.098000
25%        52.619250
50%        90.494500
75%       149.657000
max       400.846000
Name: bulk_modulus_vrh, dtype: float64


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
# pipeline creates a whole auto process
models = {
    # it outputs the mean of bulk modulus for comparison
    "1. Dummy Baseline (Mean)": Pipeline([
        ('model', DummyRegressor(strategy='mean'))
    ]),
    
    # Tier 2: Linear Baseline (Needs scaling)
    "2. Scaled Linear (Ridge)": Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0))
    ]),
    
    "3. Random Forest": Pipeline([
        ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    
    "4. Gradient Boosting (Hist)": Pipeline([
        ('model', HistGradientBoostingRegressor(learning_rate=0.1, random_state=42))
    ])
}

results = []
metrics = ['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2']

for name, pipeline in models.items():
    cv_scores = cross_validate(pipeline, X_train, y_train, cv=5, scoring=metrics, n_jobs=-1)
    
    rmse = -cv_scores['test_neg_root_mean_squared_error'].mean()
    mae = -cv_scores['test_neg_mean_absolute_error'].mean()
    r2 = cv_scores['test_r2'].mean()
    
    results.append({
        'Model': name,
        'CV RMSE': round(rmse, 3),
        'CV MAE': round(mae, 3),
        'CV R²': round(r2, 4)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


                      Model  CV RMSE  CV MAE   CV R²
   1. Dummy Baseline (Mean)   69.497  56.238 -0.0010
   2. Scaled Linear (Ridge)   32.070  21.176  0.7846
           3. Random Forest   19.279  11.772  0.9229
4. Gradient Boosting (Hist)   17.508  10.690  0.9362
